In [ ]:
import os

os.environ["LOGURU_LEVEL"] = "ERROR"


# ALM Data Pipeline Tutorial

This notebook walks through the ALM (Audio Language Model) data pipeline:
1. Read diarized audio manifests
2. Build training windows with quality filters
3. Remove overlapping windows
4. Inspect and visualize results

**No GPU required** — all stages are CPU-only. Install with `uv sync --extra audio_cpu`.

In [ ]:
import json
import os

os.environ["RAY_MAX_LIMIT_FROM_API_SERVER"] = "100000"

from nemo_curator.backends.xenna import XennaExecutor
from nemo_curator.core.client import RayClient
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.audio import ManifestReader, ManifestWriterStage
from nemo_curator.stages.audio.alm.alm_data_builder import ALMDataBuilderStage
from nemo_curator.stages.audio.alm.alm_data_overlap import ALMDataOverlapStage

## Step 1: Inspect the input data

The bundled sample has 5 audio manifest entries with diarized segments.

In [ ]:
MANIFEST_PATH = os.path.join("..", "..", "..", "tests", "fixtures", "audio", "alm", "sample_input.jsonl")
OUTPUT_DIR = "./alm_tutorial_output"

with open(MANIFEST_PATH) as f:
    entries = [json.loads(line) for line in f if line.strip()]

print(f"Input entries: {len(entries)}")
for i, entry in enumerate(entries):
    n_segments = len(entry.get("segments", []))
    sr = entry.get("audio_sample_rate", "unknown")
    speakers = len({s.get("speaker", "") for s in entry.get("segments", [])})
    print(f"  [{i}] {n_segments} segments, {speakers} speakers, {sr} Hz")

## Step 2: Run the ALM pipeline

The pipeline reads manifests, builds windows, filters overlaps, and writes output.

We run with **both** backends (Xenna and Ray Data) and compare results and timing.
Both produce identical results; wall-clock time may differ depending on data size and hardware.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, "alm_output.jsonl")


def build_pipeline() -> Pipeline:
    """Create a fresh pipeline instance (needed per run since the writer appends)."""
    p = Pipeline(name="alm_tutorial", description="Build ALM training windows from diarized manifests")
    p.add_stage(ManifestReader(manifest_path=MANIFEST_PATH))
    p.add_stage(
        ALMDataBuilderStage(
            target_window_duration=120.0,
            tolerance=0.1,
            min_sample_rate=16000,
            min_bandwidth=8000,
            min_speakers=2,
            max_speakers=5,
        )
    )
    p.add_stage(ALMDataOverlapStage(overlap_percentage=50))
    p.add_stage(ManifestWriterStage(output_path=output_path))
    return p


print(build_pipeline().describe())

In [ ]:
import time

from nemo_curator.backends.ray_data import RayDataExecutor

ray_client = RayClient()
ray_client.start()

backends = {
    "xenna": XennaExecutor,
    "ray_data": RayDataExecutor,
}

run_results = {}

for name, executor_cls in backends.items():
    if os.path.exists(output_path):
        os.remove(output_path)

    pipeline = build_pipeline()
    executor = executor_cls()

    t0 = time.time()
    pipeline.run(executor)
    elapsed = time.time() - t0

    with open(output_path) as f:
        data = [json.loads(line) for line in f if line.strip()]

    total_windows = sum(len(r.get("filtered_windows", [])) for r in data)
    total_dur = sum(r.get("filtered_dur", 0) for r in data)

    run_results[name] = {
        "time": elapsed,
        "entries": len(data),
        "windows": total_windows,
        "duration": total_dur,
        "data": data,
    }
    print(f"[{name}] {elapsed:.2f}s — {len(data)} entries, {total_windows} windows, {total_dur:.1f}s audio")

In [ ]:
MATCH_TOL = 0.1

print("\n" + "=" * 60)
print("Backend Comparison")
print("=" * 60)
print(f"{'':>12s}  {'Xenna':>10s}  {'Ray Data':>10s}  {'Match':>6s}")
print(f"{'Time (s)':>12s}  {run_results['xenna']['time']:10.2f}  {run_results['ray_data']['time']:10.2f}  {'':>6s}")
print(
    f"{'Entries':>12s}  {run_results['xenna']['entries']:10d}  {run_results['ray_data']['entries']:10d}"
    f"  {'✓' if run_results['xenna']['entries'] == run_results['ray_data']['entries'] else '✗':>6s}"
)
print(
    f"{'Windows':>12s}  {run_results['xenna']['windows']:10d}  {run_results['ray_data']['windows']:10d}"
    f"  {'✓' if run_results['xenna']['windows'] == run_results['ray_data']['windows'] else '✗':>6s}"
)
print(
    f"{'Audio (s)':>12s}  {run_results['xenna']['duration']:10.1f}  {run_results['ray_data']['duration']:10.1f}"
    f"  {'✓' if abs(run_results['xenna']['duration'] - run_results['ray_data']['duration']) < MATCH_TOL else '✗':>6s}"
)
speedup = run_results["ray_data"]["time"] / run_results["xenna"]["time"]
faster = "xenna" if speedup > 1 else "ray_data"
print(f"\n→ {faster} was {max(speedup, 1 / speedup):.1f}x faster on this dataset")

results = run_results["xenna"]["data"]

## Step 3: Inspect results

In [ ]:
print(f"Output entries: {len(results)}")

total_windows_before = sum(len(r.get("windows", [])) for r in results)
total_windows_after = sum(len(r.get("filtered_windows", [])) for r in results)
total_duration = sum(r.get("filtered_dur", 0) for r in results)

print(f"Windows before overlap filter: {total_windows_before}")
print(f"Windows after overlap filter:  {total_windows_after}")
print(f"Total filtered audio duration: {total_duration:.1f}s ({total_duration / 3600:.2f}h)")
print(
    f"Overlap reduction: {(1 - total_windows_after / total_windows_before) * 100:.0f}%"
    if total_windows_before
    else "N/A"
)

## Step 4: Understand the filtering statistics

In [ ]:
for i, r in enumerate(results):
    stats = r.get("stats", {})
    print(f"Entry [{i}]:")
    print(f"  Total segments: {stats.get('total_segments', 'N/A')}")
    print(f"  Total duration: {stats.get('total_dur', 0):.1f}s")
    print(f"  Lost (low bandwidth):    {stats.get('lost_bw', 0)}")
    print(f"  Lost (low sample rate):  {stats.get('lost_sr', 0)}")
    print(f"  Lost (speaker count):    {stats.get('lost_spk', 0)}")
    print(f"  Lost (window constraints): {stats.get('lost_win', 0)}")
    print(f"  Truncation events: {r.get('truncation_events', 0)}")
    print()

## Step 5: Visualize results

### Window duration distribution and filter effects

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Collect data from results
all_win_durs = []
all_filt_durs = []
for r in results:
    for w in r.get("windows", []):
        segs = w.get("segments", [])
        if segs:
            dur = segs[-1].get("end", 0) - segs[0].get("start", 0)
            all_win_durs.append(dur)
    all_filt_durs.extend(r.get("filtered_dur_list", []))

# 1. Window duration histogram — before vs after overlap filter
ax = axes[0, 0]
if all_win_durs:
    ax.hist(
        all_win_durs, bins=25, color="#4C72B0", alpha=0.5, label=f"Before ({len(all_win_durs)})", edgecolor="white"
    )
if all_filt_durs:
    ax.hist(
        all_filt_durs, bins=25, color="#C44E52", alpha=0.6, label=f"After ({len(all_filt_durs)})", edgecolor="white"
    )
ax.set_xlabel("Window Duration (seconds)")
ax.set_ylabel("Count")
ax.set_title("Window Durations: Before vs After Overlap Filter")
ax.legend()

# 2. Speaker count distribution across filtered windows
ax = axes[0, 1]
speaker_counts = []
for r in results:
    for fw in r.get("filtered_windows", []):
        speakers = {s.get("speaker", "") for s in fw.get("segments", [])}
        speaker_counts.append(len(speakers))
if speaker_counts:
    counts = Counter(speaker_counts)
    labels = sorted(counts.keys())
    ax.bar([str(k) for k in labels], [counts[k] for k in labels], color="#55A868", edgecolor="white")
ax.set_xlabel("Number of Speakers")
ax.set_ylabel("Number of Windows")
ax.set_title("Speaker Count per Filtered Window")

# 3. Filter loss breakdown — stacked bar per entry
ax = axes[1, 0]
entry_labels = [f"Entry {i}" for i in range(len(results))]
loss_bw = [r.get("stats", {}).get("lost_bw", 0) for r in results]
loss_sr = [r.get("stats", {}).get("lost_sr", 0) for r in results]
loss_spk = [r.get("stats", {}).get("lost_spk", 0) for r in results]
loss_win = [r.get("stats", {}).get("lost_win", 0) for r in results]
x = np.arange(len(results))
w = 0.5
ax.bar(x, loss_bw, w, label="Bandwidth", color="#4C72B0")
ax.bar(x, loss_sr, w, bottom=loss_bw, label="Sample Rate", color="#DD8452")
bottoms = [a + b for a, b in zip(loss_bw, loss_sr, strict=True)]
ax.bar(x, loss_spk, w, bottom=bottoms, label="Speaker Count", color="#55A868")
bottoms2 = [a + b for a, b in zip(bottoms, loss_spk, strict=True)]
ax.bar(x, loss_win, w, bottom=bottoms2, label="Window Constraint", color="#C44E52")
ax.set_xticks(x)
ax.set_xticklabels(entry_labels, fontsize=8)
ax.set_ylabel("Segments Lost")
ax.set_title("Filter Loss Breakdown by Entry")
ax.legend(fontsize=8)

# 4. Per-entry window yield — before vs after
ax = axes[1, 1]
wins_before = [len(r.get("windows", [])) for r in results]
wins_after = [len(r.get("filtered_windows", [])) for r in results]
x = np.arange(len(results))
w = 0.35
ax.bar(x - w / 2, wins_before, w, label="Before Overlap Filter", color="#4C72B0", edgecolor="white")
ax.bar(x + w / 2, wins_after, w, label="After Overlap Filter", color="#C44E52", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(entry_labels, fontsize=8)
ax.set_ylabel("Number of Windows")
ax.set_title("Window Yield per Entry")
ax.legend(fontsize=8)

fig.suptitle(
    f"ALM Pipeline — {len(results)} entries, {sum(wins_before)} → {sum(wins_after)} windows", fontsize=13, y=1.01
)
fig.tight_layout()
plt.show()

print(
    f"\nFiltered durations — min: {min(all_filt_durs):.1f}s, max: {max(all_filt_durs):.1f}s, "
    f"mean: {np.mean(all_filt_durs):.1f}s"
    if all_filt_durs
    else "No filtered windows."
)

## Step 6: Experiment with parameters

Try different overlap thresholds and see how they affect the output:

In [ ]:
from nemo_curator.tasks.audio_task import AudioTask

sample_entry = entries[0]
task = AudioTask(data=sample_entry)

builder = ALMDataBuilderStage(
    target_window_duration=120.0,
    tolerance=0.1,
    min_sample_rate=16000,
    min_bandwidth=8000,
    min_speakers=2,
    max_speakers=5,
)
built = builder.process(task)
n_windows = len(built.data.get("windows", []))

print(f"Windows from entry[0]: {n_windows}")
print()

for pct in [0, 25, 50, 75, 100]:
    overlap = ALMDataOverlapStage(overlap_percentage=pct)
    filtered = overlap.process(built)
    n_filtered = len(filtered.data.get("filtered_windows", []))
    print(
        f"  overlap_percentage={pct:3d}: {n_filtered:3d} windows kept ({n_filtered / n_windows * 100:.0f}%)"
        if n_windows
        else "  No windows"
    )

## Cleanup

Shut down the Ray cluster started by `RayClient`.

In [ ]:
ray_client.stop()